In [15]:
from collections import Counter
from typing import Union, Literal, Callable

import pandas as pd
import numpy as np
import sklearn as sk
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from sklearn.model_selection import train_test_split

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


ds = load_dataset("sh0416/ag_news")
train_df = pd.DataFrame(ds["train"])
test_df = pd.DataFrame(ds["test"])


In [14]:
print("Example Training Entry:")
print(train_df.head(1))
print("\nDistribution of Training Labels:")
print(train_df.groupby("label").size())

print("\nExample Testing Entry:")
print(test_df.head(1))
print("\nDistribution of Testing Labels:")
print(test_df.groupby("label").size())

Example Training Entry:
   label                                              title  \
0      3  Wall St. Bears Claw Back Into the Black (Reuters)   

                                         description  
0  Reuters - Short-sellers, Wall Street's dwindli...  

Distribution of Training Labels:
label
1    30000
2    30000
3    30000
4    30000
dtype: int64

Example Testing Entry:
   label                              title  \
0      3  Fears for T N pension after talks   

                                         description  
0  Unions representing workers at Turner   Newall...  

Distribution of Testing Labels:
label
1    1900
2    1900
3    1900
4    1900
dtype: int64


In [27]:
total_train_np = train_df.to_numpy()
train, val = train_test_split(
    total_train_np,
    test_size=0.1,
    random_state=SEED,
)
test = test_df.to_numpy()
print(train.shape, val.shape, test.shape)

train_examples = train[:, 1:]
train_labels = train[:, 0]
val_examples = val[:, 1:]
val_labels = val[:, 0]
test_examples = test[:, 1:]
test_labels = test[:, 0]

print("Num Training Examples:", len(train))
print("Num Validation Examples:", len(val))
print("Num Testing Examples:", len(test))


(108000, 3) (12000, 3) (7600, 3)
Num Training Examples: 108000
Num Validation Examples: 12000
Num Testing Examples: 7600


In [28]:
from torch.utils.data import dataset
from typing import Union

# should already be tokenized
class NewsDataset(dataset.Dataset):
    def __init__(self, examples: Union[torch.Tensor, np.ndarray], labels: Union[torch.Tensor, np.ndarray]):
        self.examples = examples if isinstance(examples, torch.Tensor) else torch.Tensor(examples)
        self.labels = labels if isinstance(labels, torch.Tensor) else torch.Tensor(labels)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx], self.labels[idx]

In [48]:
import re
from collections import Counter

class Tokenizer:
    def __init__(self):
        # token to index (basically same as one hot
        self.vocab = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.vocab_length = 4

        # inverse for debugging and visualization
        self.inverse = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
        self.initialized = False

    def _entry_to_tokens(self, entry):
        text = entry[0] + " " + entry[1]
        text = str(text).lower()
        tokens = re.findall(r"[a-z0-9']+", text)
        return tokens

    def build_vocab(self, examples: Union[torch.Tensor, np.ndarray]):
        if isinstance(examples, torch.Tensor):
            examples = examples.numpy()

        counts = Counter()

        # each example has title, desc
        for example in examples:
            # join title and description, lowercase, and extract tokens
            tokens = self._entry_to_tokens(example)
            counts.update(tokens)

        for token, freq in counts.items():
            if freq > 1 and token not in self.vocab:
                self.vocab[token] = len(self.inverse)
                self.inverse.append(token)
                self.vocab_length += 1

        self.initialized = True

    def lookup(self, word: str) -> int:
        if word not in self.vocab:
            return self.vocab["<UNK>"]
        return self.vocab[word]

    def tokenize_entries(self, entries: Union[torch.Tensor, np.ndarray], max_len = 128):
        if isinstance(entries, torch.Tensor):
            entries = entries.numpy()
        if entries.ndim == 1:
            entries = np.expand_dims(entries, axis=0)
        scaffold = np.zeros((len(entries), max_len), dtype="int8")
        sequences = np.full_like(scaffold, self.vocab["<PAD>"], dtype="int32")
        lengths = np.zeros(len(entries), dtype="int8")
        sequences[:, 0] = self.vocab["<SOS>"]
        for i, entry in enumerate(entries):
            tokens = self._entry_to_tokens(entry)
            lengths[i] = min(len(tokens)+2, max_len)
            for j, token in enumerate(tokens[:max_len-2]):
                sequences[i, j+1] = self.lookup(token)
            sequences[i, min(len(tokens)+1, max_len-1)] = self.vocab["<EOS>"]
        return sequences, lengths




In [49]:
tokenizer = Tokenizer()
tokenizer.build_vocab(train_examples)
print("Vocab Size:", len(tokenizer.vocab))
test_ex = train_examples[0]
test_seq, _ = tokenizer.tokenize_entries(train_examples[0])
test_inverse = tokenizer.inverse
print("Example Entry:", test_ex)
print("Tokenized Sequence:", test_seq)
for token in test_seq[0]:
    print(test_inverse[token], end="")


Vocab Size: 44335
Example Entry: ['Early Beatles, U.S. Style'
 'Fans who grew up listening to Americanized versions of Beatles\' albums will find guilty pleasure in "The Capitol Albums, Vol. 1."']
Tokenized Sequence: [[ 1  4  5  6  7  8  9 10 11 12 13 14  3 15 16  3 17 18 19 20 21 22 23 24
  17 25 26  2  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0  0]]
<SOS>earlybeatlesusstylefanswhogrewuplisteningto<UNK>versionsof<UNK>albumswillfindguiltypleasureinthecapitolalbumsvol1<EOS><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><P

In [52]:
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from dataclasses import dataclass
from typing import Callable

def mean_pool(lstm_out, lengths):
    # lstm_out: (batch_size, seq_len, hidden_dim)
    # lengths: (batch_size,)
    batch_size, seq_len, hidden_dim = lstm_out.shape
    lengths = lengths.to(lstm_out.device)
    mask = torch.arange(seq_len, device=lstm_out.device).expand(batch_size, seq_len) < lengths.unsqueeze(1)
    masked_out = lstm_out * mask.unsqueeze(2) # (batch_size, seq_len, hidden_dim) with masked positions zeroed out
    pooled = masked_out.sum(dim=1) / lengths.float().unsqueeze(1) # (batch_size, hidden_dim)
    return pooled

def max_pool(lstm_out, lengths):
    batch_size, seq_len, hidden_dim = lstm_out.shape
    lengths = lengths.to(lstm_out.device)
    mask = torch.arange(seq_len, device=lstm_out.device).expand(batch_size, seq_len) < lengths.unsqueeze(1)
    masked_out = lstm_out.masked_fill(~mask.unsqueeze(2), float('-inf')) # (batch_size, seq_len, hidden_dim) with masked positions set to -inf
    pooled, _ = masked_out.max(dim=1) # (batch_size, hidden_dim)
    return pooled

def last_pool(lstm_out, lengths):
    batch_size, seq_len, hidden_dim = lstm_out.shape
    lengths = lengths.to(lstm_out.device)
    last_indices = (lengths - 1).unsqueeze(1).unsqueeze(2).expand(batch_size, 1, hidden_dim) # (batch_size, 1, hidden_dim)
    pooled = lstm_out.gather(1, last_indices).squeeze(1) # (batch_size, hidden_dim)
    return pooled

@dataclass
class Hyperparams:
    embedding_dim: int = 128
    hidden_dim: int = 256
    num_layers: int = 2
    bidirectional: bool = False
    dropout: float = 0.2
    pooling: Callable = mean_pool

class Model(nn.Module):
    def __init__(self, tokenizer: Tokenizer, hyperparams: Hyperparams):
        super().__init__()
        self.embed = nn.Embedding(tokenizer.vocab_length, hyperparams.embedding_dim)
        self.lstm = nn.LSTM(
            input_size=hyperparams.embedding_dim,
            hidden_size=hyperparams.hidden_dim,
            num_layers=hyperparams.num_layers,
            bidirectional=hyperparams.bidirectional,
            dropout=hyperparams.dropout if hyperparams.num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.pooling = hyperparams.pooling
        self.dropout = nn.Dropout(hyperparams.dropout)
        self.out = nn.Linear(hyperparams.hidden_dim * (2 if hyperparams.bidirectional else 1), 4)

    def forward(self, x, lengths):
        embedded = self.embed(x) # (batch_size, seq_len, embedding_dim)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        lstm_out, _ = pad_packed_sequence(packed_out, batch_first=True) # (batch_size, seq_len, hidden_dim * num_directions)
        pooled = self.pooling(lstm_out, lengths) # (batch_size, hidden_dim * num_directions)
        pooled = self.dropout(pooled)
        logits = self.out(pooled) # (batch_size, num_classes)
        return logits